In [2]:
import os
import mysql.connector
from dotenv import load_dotenv

load_dotenv()

def get_connection():
    return mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        database=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        ssl_ca=os.getenv("DB_SSL_CA")
    )

conn = get_connection()
conn.close()

In [3]:
import pandas as pd
import google.genai as genai
from tabulate import tabulate
from IPython.display import display


In [4]:
from google import genai

client = genai.Client(api_key=os.getenv("LLM_API_KEY"))

In [37]:
SYSTEM_PROMPT = """
Sos un experto en bases de datos MySQL. Tu única tarea es convertir preguntas 
en español a consultas SQL válidas para la base de datos 'biblioia'.

REGLAS:
- Respondé ÚNICAMENTE con la consulta SQL, sin explicaciones ni comentarios.
- No uses bloques de código ni backticks.
- Preferí usar las VISTAS disponibles cuando corresponda.
- Si la pregunta no se puede responder con el esquema dado, respondé: 
  SELECT 'No puedo responder esa pregunta con los datos disponibles';

=== ESQUEMA ===

GENERO (id_genero INT PK, nombre VARCHAR(60) UNIQUE NOT NULL, descripcion VARCHAR(255))
AUTOR (id_autor INT PK, nombre VARCHAR(80) NOT NULL, apellido VARCHAR(80) NOT NULL, nacionalidad VARCHAR(60))
LIBRO (isbn VARCHAR(20) PK, titulo VARCHAR(200) NOT NULL, anio_publicacion YEAR, stock_total SMALLINT, stock_disponible SMALLINT)
  -- stock_disponible <= stock_total siempre
LIBRO_AUTOR (isbn FK->LIBRO, id_autor FK->AUTOR) -- N:M
LIBRO_GENERO (isbn FK->LIBRO, id_genero FK->GENERO) -- N:M
SOCIO (id_socio INT PK, dni VARCHAR(15) UNIQUE, nombre VARCHAR(80), apellido VARCHAR(80), email VARCHAR(120) UNIQUE, fecha_alta DATE, estado VARCHAR(12))
  -- estado: 'ACTIVO', 'SUSPENDIDO', 'BAJA'
EJEMPLAR (id_ejemplar INT PK, isbn FK->LIBRO, nro_ejemplar SMALLINT, estado_fisico VARCHAR(12))
  -- estado_fisico: 'BUENO', 'DETERIORADO', 'BAJA'
PRESTAMO (id_prestamo INT PK, id_socio FK->SOCIO, id_ejemplar FK->EJEMPLAR, fecha_prestamo DATE, fecha_vencimiento DATE, fecha_devolucion DATE NULL, estado VARCHAR(12))
  -- estado: 'ACTIVO', 'DEVUELTO', 'VENCIDO'
SANCION (id_sancion INT PK, id_socio FK->SOCIO, tipo VARCHAR(20), fecha_inicio DATE, fecha_fin DATE, motivo VARCHAR(255))
  -- tipo: 'MORA', 'DAÑO', 'PERDIDA', 'OTRO'
  -- activa cuando fecha_fin >= CURRENT_DATE
AUDITORIA_PRESTAMOS (id_audit INT PK, id_prestamo INT, operacion VARCHAR(10), estado_nuevo VARCHAR(12), estado_viejo VARCHAR(12), usuario_bd VARCHAR(80), fecha_hora DATETIME)

=== VISTAS DISPONIBLES ===

v_prestamos_vencidos (id_prestamo, dni, socio, titulo, fecha_prestamo, fecha_vencimiento, dias_de_mora)
  -- préstamos con estado='VENCIDO' y sin devolución

v_prestamos_activos (id_prestamo, id_socio, dni, socio, email, isbn, titulo, nro_ejemplar, fecha_prestamo, fecha_vencimiento, dias_vencido)
  -- todos los préstamos con estado='ACTIVO'

v_libros_disponibles (isbn, titulo, stock_disponible, generos, autores)
  -- libros con stock_disponible > 0 y al menos un ejemplar en buen estado

v_historial_socios (id_socio, dni, socio, isbn, titulo, fecha_prestamo, fecha_vencimiento, fecha_devolucion, estado_prestamo)
  -- historial completo de préstamos por socio

v_libros_mas_prestados (isbn, titulo, autores, total_prestamos)
  -- ranking de libros por cantidad de préstamos

v_socios_sancionados (id_socio, dni, socio, estado, tipo, fecha_inicio, fecha_fin, motivo, dias_restantes)
  -- socios con sanciones activas hoy

v_autores_prolíficos (id_autor, autor, nacionalidad, cantidad_libros)
  -- autores con más de 1 libro en la biblioteca

v_lista_socios (lista_socios)
-- contine todas las filas de la tabla SOCIO

v_reporte_salud_biblioteca (total_inventario, ejemplares_circulacion, tasa_ocupacion, total_socios, porcentaje_socios_sancionados)
  -- contiene una única fila con las métricas generales y el estado de salud en tiempo real de la biblioteca

=== EJEMPLOS ===

Pregunta: ¿Cuáles son los 5 libros más prestados este año?
SQL: SELECT isbn, titulo, total_prestamos FROM v_libros_mas_prestados LIMIT 5;

Pregunta: ¿Qué socios tienen préstamos vencidos en este momento?
SQL: SELECT DISTINCT dni, socio FROM v_prestamos_vencidos;

Pregunta: ¿Qué libros de ciencia ficción están disponibles para prestar?
SQL: SELECT isbn, titulo, stock_disponible FROM v_libros_disponibles WHERE generos LIKE '%Ciencia Ficción%';
"""

In [6]:
def text_to_sql(pregunta: str) -> str:
    respuesta = client.models.generate_content(
       model="gemini-2.5-flash",
        contents=SYSTEM_PROMPT + f"\nPregunta: {pregunta}\nSQL:"
    )
    sql = respuesta.text.strip()
    # Por las dudas, limpiamos backticks que Gemini a veces agrega
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql


In [7]:
def ejecutar_consulta(sql: str) -> pd.DataFrame:
    conn = get_connection()
    try:
        df = pd.read_sql(sql, conn)
        return df
    except Exception as e:
        return pd.DataFrame({"Error": [str(e)]})
    # cirra la conexion 
    finally:
        conn.close()


In [8]:
def agente_responder(pregunta: str, mostrar_sql: bool = True):
    print(f"\n Pregunta: {pregunta}")
    print("-" * 60)
    
    sql = text_to_sql(pregunta)
    
    if mostrar_sql:
        print(f" SQL generado:\n{sql}")
        print("-" * 60)
    
    df = ejecutar_consulta(sql)
    
    if df.empty:
        print(" La consulta no devolvió resultados.")
    else:
        print(f" Resultado ({len(df)} filas):")
        display(df)
    
    return df

In [9]:
agente_responder("¿Cuáles son los 5 libros más prestados este año?")


 Pregunta: ¿Cuáles son los 5 libros más prestados este año?
------------------------------------------------------------
 SQL generado:
SELECT isbn, titulo, total_prestamos FROM v_libros_mas_prestados LIMIT 5;
------------------------------------------------------------


C:\Users\Millenium\AppData\Local\Temp\ipykernel_2132\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (5 filas):


,isbn,titulo,total_prestamos
0,978-0-09-954793-8,Sapiens,4
1,978-0-13-235088-4,Clean Code,3
2,978-950-724-528-3,El Aleph,3
3,978-950-724-100-1,Ficciones,3
4,978-0-679-72020-1,¿Sueñan los androides con ovejas eléctricas?,3


,isbn,titulo,total_prestamos
0,978-0-09-954793-8,Sapiens,4
1,978-0-13-235088-4,Clean Code,3
2,978-950-724-528-3,El Aleph,3
3,978-950-724-100-1,Ficciones,3
4,978-0-679-72020-1,¿Sueñan los androides con ovejas eléctricas?,3


In [9]:
agente_responder("Qué socios tienen préstamos vencidos en este momento?")


 Pregunta: Qué socios tienen préstamos vencidos en este momento?
------------------------------------------------------------
 SQL generado:
SELECT DISTINCT dni, socio FROM v_prestamos_vencidos;
------------------------------------------------------------


C:\Users\Millenium\AppData\Local\Temp\ipykernel_15388\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (10 filas):


,dni,socio
0,30100023,Benjamín Gutiérrez
1,30100024,Renata Mendoza
2,30100025,Axel Silva
3,30100026,Brenda Cabrera
4,30100027,Rodrigo Aguilar
5,30100028,Zoe Vega
6,30100029,Sebastián Reyes
7,30100001,Lucas Fernández
8,30100002,Valentina Gómez
9,30100003,Matías Rodríguez


,dni,socio
0,30100023,Benjamín Gutiérrez
1,30100024,Renata Mendoza
2,30100025,Axel Silva
3,30100026,Brenda Cabrera
4,30100027,Rodrigo Aguilar
5,30100028,Zoe Vega
6,30100029,Sebastián Reyes
7,30100001,Lucas Fernández
8,30100002,Valentina Gómez
9,30100003,Matías Rodríguez


In [10]:
agente_responder("¿Cuántos ejemplares disponibles hay del libro con ISBN 978-0-13-235088-4?")


 Pregunta: ¿Cuántos ejemplares disponibles hay del libro con ISBN 978-0-13-235088-4?
------------------------------------------------------------
 SQL generado:
SELECT stock_disponible FROM v_libros_disponibles WHERE isbn = '978-0-13-235088-4';
------------------------------------------------------------


C:\Users\Millenium\AppData\Local\Temp\ipykernel_15388\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (1 filas):


,stock_disponible
0,3


,stock_disponible
0,3


In [11]:
agente_responder("¿Qué libros de ciencia ficción están disponibles para prestar?")


 Pregunta: ¿Qué libros de ciencia ficción están disponibles para prestar?
------------------------------------------------------------
 SQL generado:
SELECT isbn, titulo, stock_disponible FROM v_libros_disponibles WHERE generos LIKE '%Ciencia Ficción%';
------------------------------------------------------------


C:\Users\Millenium\AppData\Local\Temp\ipykernel_15388\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (7 filas):


,isbn,titulo,stock_disponible
0,978-0-06-112008-4,La mano izquierda de la oscuridad,1
1,978-0-14-118776-1,Los desposeídos,1
2,978-0-345-34314-2,2001: Una odisea del espacio,2
3,978-0-345-39180-3,Cita con Rama,2
4,978-0-553-29335-7,Fundación,2
5,978-0-553-38316-3,El fin de la eternidad,2
6,978-0-679-72020-1,¿Sueñan los androides con ovejas eléctricas?,1


,isbn,titulo,stock_disponible
0,978-0-06-112008-4,La mano izquierda de la oscuridad,1
1,978-0-14-118776-1,Los desposeídos,1
2,978-0-345-34314-2,2001: Una odisea del espacio,2
3,978-0-345-39180-3,Cita con Rama,2
4,978-0-553-29335-7,Fundación,2
5,978-0-553-38316-3,El fin de la eternidad,2
6,978-0-679-72020-1,¿Sueñan los androides con ovejas eléctricas?,1


In [12]:
agente_responder("¿Cuál es el historial de préstamos del socio con DNI 30100001?")


 Pregunta: ¿Cuál es el historial de préstamos del socio con DNI 30100001?
------------------------------------------------------------
 SQL generado:
SELECT isbn, titulo, fecha_prestamo, fecha_vencimiento, fecha_devolucion, estado_prestamo FROM v_historial_socios WHERE dni = '30100001'
------------------------------------------------------------


C:\Users\Millenium\AppData\Local\Temp\ipykernel_15388\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (3 filas):


,isbn,titulo,fecha_prestamo,fecha_vencimiento,fecha_devolucion,estado_prestamo
0,978-0-553-29335-7,Fundación,2026-05-15,2026-05-29,None,ACTIVO
1,978-0-553-38316-3,El fin de la eternidad,2026-04-15,2026-04-29,None,VENCIDO
2,978-0-553-29335-7,Fundación,2026-01-05,2026-01-19,2026-01-18,DEVUELTO


,isbn,titulo,fecha_prestamo,fecha_vencimiento,fecha_devolucion,estado_prestamo
0,978-0-553-29335-7,Fundación,2026-05-15,2026-05-29,None,ACTIVO
1,978-0-553-38316-3,El fin de la eternidad,2026-04-15,2026-04-29,None,VENCIDO
2,978-0-553-29335-7,Fundación,2026-01-05,2026-01-19,2026-01-18,DEVUELTO


In [13]:
agente_responder("¿Qué autores tienen más de 3 libros en la biblioteca?")


 Pregunta: ¿Qué autores tienen más de 3 libros en la biblioteca?
------------------------------------------------------------
 SQL generado:
SELECT autor, cantidad_libros FROM v_autores_prolíficos WHERE cantidad_libros > 3;
------------------------------------------------------------


C:\Users\Millenium\AppData\Local\Temp\ipykernel_15388\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (1 filas):


,autor,cantidad_libros
0,Jorge Luis Borges,4


,autor,cantidad_libros
0,Jorge Luis Borges,4


In [14]:
agente_responder("¿Cuántas sanciones se generaron en el último mes?")


 Pregunta: ¿Cuántas sanciones se generaron en el último mes?
------------------------------------------------------------
 SQL generado:
SELECT COUNT(*) FROM SANCION WHERE fecha_inicio >= DATE_SUB(CURDATE(), INTERVAL 1 MONTH)
------------------------------------------------------------


C:\Users\Millenium\AppData\Local\Temp\ipykernel_15388\1564992065.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


 Resultado (1 filas):


,COUNT(*)
0,1


,COUNT(*)
0,1


In [40]:
agente_responder("cual es el genero preferido por el socio con DNI 30100002 ")


 Pregunta: cual es el genero preferido por el socio con DNI 30100002 
------------------------------------------------------------


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 34.740836303s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},

In [38]:
agente_responder("Dame un reporte del estado de salud actual de la biblioteca")


 Pregunta: Dame un reporte del estado de salud actual de la biblioteca
------------------------------------------------------------


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 46.8089158s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, '

In [39]:
agente_responder("cuál es el total de socios?")


 Pregunta: cuál es el total de socios?
------------------------------------------------------------


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 45.843430004s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'},